# Conditional DCGAN on MNIST (PyTorch)

A **Conditional GAN (cGAN)**: unlike a plain GAN that makes random digits, this one lets you **choose which digit to generate** (e.g. "draw me a 7").

How: the digit **label** (0–9) is fed into *both* networks.
- **Generator** gets `noise + label` → learns to draw the requested digit.
- **Discriminator** gets `image + label` → learns "is this a real image *of that digit*?"

It's a DCGAN backbone (BCE loss, no gradient penalty), so it's fast and stable.

Reference: Mirza & Osindero, *Conditional Generative Adversarial Nets* (2014).

## 1. Setup & imports

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Hyperparameters

In [ ]:
config = dict(
    image_size=32,
    channels=1,
    num_classes=10,     # digits 0-9
    z_dim=100,
    embed_size=100,     # size of the label embedding fed to the generator
    g_features=64,
    d_features=64,
    batch_size=128,
    epochs=10,
    lr=2e-4,
    beta1=0.5,
    beta2=0.999,
    seed=42,
)

torch.manual_seed(config["seed"])

## 3. Data

In [ ]:
transform = transforms.Compose([
    transforms.Resize(config["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,  # 0 on Windows/Jupyter
    pin_memory=True,
    drop_last=True,
)
print(f"Dataset size: {len(dataset)} | batches/epoch: {len(dataloader)}")

## 4. Models

**Generator**: the label is turned into an embedding vector and concatenated with the noise vector, so the input carries *which digit to draw*.

**Discriminator**: the label is turned into a full-size channel and stacked onto the image, so it judges *is this a real image of this specific digit?*

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, channels, g, num_classes, embed_size):
        super().__init__()
        self.embed = nn.Embedding(num_classes, embed_size)
        self.net = nn.Sequential(
            # (z_dim + embed_size) x 1 x 1 -> (g*4) x 4 x 4
            self._block(z_dim + embed_size, g * 4, 4, 1, 0),
            self._block(g * 4, g * 2, 4, 2, 1),   # -> 8x8
            self._block(g * 2, g, 4, 2, 1),       # -> 16x16
            nn.ConvTranspose2d(g, channels, 4, 2, 1),  # -> 32x32
            nn.Tanh(),
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, noise, labels):
        # noise: (N, z_dim, 1, 1); labels: (N,)
        emb = self.embed(labels).unsqueeze(-1).unsqueeze(-1)  # (N, embed_size, 1, 1)
        x = torch.cat([noise, emb], dim=1)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, channels, d, num_classes, image_size):
        super().__init__()
        self.image_size = image_size
        self.embed = nn.Embedding(num_classes, image_size * image_size)
        self.net = nn.Sequential(
            # (channels + 1) x 32 x 32 -> d x 16 x 16
            nn.Conv2d(channels + 1, d, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            self._block(d, d * 2, 4, 2, 1),       # -> 8x8
            self._block(d * 2, d * 4, 4, 2, 1),   # -> 4x4
            nn.Conv2d(d * 4, 1, 4, 1, 0),         # -> 1x1
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x, labels):
        # turn the label into an extra image channel and stack it on the input
        emb = self.embed(labels).view(labels.size(0), 1, self.image_size, self.image_size)
        x = torch.cat([x, emb], dim=1)
        return self.net(x).view(-1)


def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
gen = Generator(config["z_dim"], config["channels"], config["g_features"],
                config["num_classes"], config["embed_size"]).to(device)
disc = Discriminator(config["channels"], config["d_features"],
                     config["num_classes"], config["image_size"]).to(device)
gen.apply(weights_init)
disc.apply(weights_init)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator params:     {n_params(gen):,}")
print(f"Discriminator params: {n_params(disc):,}")

## 5. Loss, optimizers & wandb

In [ ]:
criterion = nn.BCEWithLogitsLoss()
opt_gen = optim.Adam(gen.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))
opt_disc = optim.Adam(disc.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))

# Fixed noise + fixed labels (0..9 repeated) so each epoch's preview shows the same digits in order.
fixed_noise = torch.randn(80, config["z_dim"], 1, 1, device=device)
fixed_labels = torch.arange(10, device=device).repeat(8)  # 0..9, 0..9, ... (80 total)

wandb.init(project="cgan-mnist", config=config)

## 6. Training loop

In [ ]:
step = 0
for epoch in range(config["epochs"]):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{config['epochs']}", leave=True)
    for real, labels in pbar:
        real = real.to(device)
        labels = labels.to(device)
        bs = real.size(0)
        noise = torch.randn(bs, config["z_dim"], 1, 1, device=device)
        fake = gen(noise, labels)

        # ---- Train Discriminator ----
        disc_real = disc(real, labels)
        loss_disc_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake.detach(), labels)
        loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_disc = (loss_disc_real + loss_disc_fake) / 2

        opt_disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # ---- Train Generator ----
        output = disc(fake, labels)
        loss_gen = criterion(output, torch.ones_like(output))

        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

        if step % 50 == 0:
            pbar.set_postfix(loss_d=f"{loss_disc.item():.3f}", loss_g=f"{loss_gen.item():.3f}")
            wandb.log({"loss/discriminator": loss_disc.item(),
                       "loss/generator": loss_gen.item(),
                       "epoch": epoch}, step=step)
        step += 1

    # ---- End of epoch: preview a grid of 0..9 (each column is a digit class) ----
    gen.eval()
    with torch.no_grad():
        samples = gen(fixed_noise, fixed_labels)
    gen.train()
    grid = make_grid(samples, nrow=10, normalize=True, value_range=(-1, 1))
    wandb.log({"generated (cols = 0..9)": wandb.Image(grid, caption=f"epoch {epoch + 1}")}, step=step)

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(gen.state_dict(), f"checkpoints/cgan_gen_ep{epoch + 1}.pth")

print("Training complete.")

## 7. Save & finish

In [ ]:
torch.save(gen.state_dict(), "checkpoints/cgan_generator.pth")
wandb.finish()
print("Saved to ./checkpoints/cgan_generator.pth")

## 8. Generate a specific digit on demand 🎯

Change `DIGIT` to whatever you want (0–9) and run the cell.

In [ ]:
import matplotlib.pyplot as plt

DIGIT = 7  # <-- change me to any digit 0..9

gen.eval()
with torch.no_grad():
    noise = torch.randn(64, config["z_dim"], 1, 1, device=device)
    labels = torch.full((64,), DIGIT, dtype=torch.long, device=device)
    samples = gen(noise, labels)
grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1)).cpu()

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title(f"cGAN: 64 generated '{DIGIT}'s")
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.show()

## 9. Generate all digits 0–9 in order

In [ ]:
gen.eval()
with torch.no_grad():
    noise = torch.randn(100, config["z_dim"], 1, 1, device=device)
    labels = torch.arange(10, device=device).repeat_interleave(10)  # 0000000000 1111111111 ...
    samples = gen(noise, labels)
grid = make_grid(samples, nrow=10, normalize=True, value_range=(-1, 1)).cpu()

plt.figure(figsize=(10, 10))
plt.axis("off")
plt.title("cGAN: each row is one digit class (0 top -> 9 bottom)")
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.show()

## Generate any digit (standalone)
Set `DIGIT` to the number you want (0-9) and run. This rebuilds the generator and loads the saved weights, so it works even on a fresh kernel (just run the imports + `Generator` class cell first).

In [ ]:
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

DIGIT = 7          # <-- the number you want (0..9)
HOW_MANY = 64      # how many to generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# rebuild generator and load trained weights (run the Generator-class cell first)
gen = Generator(z_dim=100, channels=1, g=64, num_classes=10, embed_size=100).to(device)
gen.load_state_dict(torch.load("checkpoints/cgan_generator.pth", map_location=device))
gen.eval()

with torch.no_grad():
    noise = torch.randn(HOW_MANY, 100, 1, 1, device=device)
    labels = torch.full((HOW_MANY,), DIGIT, dtype=torch.long, device=device)
    samples = gen(noise, labels)

grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1)).cpu()
plt.figure(figsize=(8, 8)); plt.axis("off")
plt.title(f"cGAN: {HOW_MANY} generated '{DIGIT}'s")
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.show()